In [1]:
import numpy as np
import pandas as pd
from pyfaidx import Fasta
import gzip
import sys
import os

In [2]:
def load_repeatmasker_data(rmsk_file):
    """Load RepeatMasker data and identify SINE B2 elements."""
    print("\nLoading RepeatMasker data...")
    
    if rmsk_file.endswith('.gz'):
        df = pd.read_csv(rmsk_file, sep='\t', compression='gzip', header=None)
    else:
        df = pd.read_csv(rmsk_file, sep='\t', header=None)
    
    print(f"Loaded {len(df)} repeat annotations")
    print(f"Columns: {df.shape[1]}")
    print(f"\nFirst few rows:")
    print(df.head())
    
    # UCSC rmsk format
    if df.shape[1] >= 17:
        df.columns = [
            'bin', 'swScore', 'milliDiv', 'milliDel', 'milliIns',
            'genoName', 'genoStart', 'genoEnd', 'genoLeft', 'strand',
            'repName', 'repClass', 'repFamily', 'repStart', 'repEnd', 'repLeft', 'id'
        ]
    
    print(f"\nRepeat classes found:")
    if 'repClass' in df.columns:
        print(df['repClass'].value_counts().head(10))
    
    print(f"\nRepeat names containing 'B2':")
    if 'repName' in df.columns:
        b2_names = df[df['repName'].str.contains('B2', case=False, na=False)]['repName'].value_counts()
        print(b2_names.head(20))
    
    return df

In [3]:
RMSK_FILE = "/project2/fudenber_735/genomes/mm10/database/rmsk.txt.gz"
GENOME_FASTA = "/project2/fudenber_735/genomes/mm10/mm10.fa"

In [4]:
rmsk_df = load_repeatmasker_data(RMSK_FILE)


Loading RepeatMasker data...
Loaded 5333739 repeat annotations
Columns: 17

First few rows:
    0      1    2   3    4     5        6        7          8  9        10  \
0  607  12955  105   9   10  chr1  3000000  3002128 -192469843  -  L1_Mus3   
1  607   1216  268  31  105  chr1  3003152  3003994 -192467977  -   L1Md_F   
2  607    234  279   0    0  chr1  3003993  3004054 -192467917  -  L1_Mus3   
3  607   3685  199  21   14  chr1  3004040  3004206 -192467765  +   L1_Rod   
4  607    376   62  31    0  chr1  3004206  3004270 -192467701  +  (CAAA)n   

              11             12    13    14    15  16  
0           LINE             L1 -3055  3592  1466   1  
1           LINE             L1 -5902   617     1   2  
2           LINE             L1 -6034   297   237   3  
3           LINE             L1  1321  1492 -4355   4  
4  Simple_repeat  Simple_repeat     4    69     0   5  

Repeat classes found:
repClass
SINE              1600329
Simple_repeat     1060618
LINE              

In [ ]:
rmsk_df.columns

In [5]:
sineB2 = rmsk_df[rmsk_df['repName'] == 'B2_Mm2'].reset_index()

In [6]:
sineB2 = sineB2.rename(columns={"index": "rmsk_index"})

In [7]:
ctcf = pd.read_csv(
    "/project2/fudenber_735/motifs/mm10/jaspar/MA0139.1.tsv.gz",
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "name", "score", "score2", "strand"],
)

In [8]:
# Ensure correct types
ctcf["start"] = ctcf["start"].astype(int)
ctcf["end"] = ctcf["end"].astype(int)

In [9]:
import bioframe

In [10]:
overlap = bioframe.overlap(
    sineB2,
    ctcf,
    how="left",
    suffixes=("", "_ctcf"),
    cols1=("genoName", "genoStart", "genoEnd"),
    cols2=("chrom", "start", "end"),
    return_index=True,
)

In [ ]:
overlap

In [11]:
has_ctcf_mask = overlap["chrom_ctcf"].notna().values

sineB2_with_ctcf = sineB2.loc[overlap.loc[has_ctcf_mask, "index"]].copy()
sineB2_no_ctcf = sineB2.loc[overlap.loc[~has_ctcf_mask, "index"]].copy()

In [12]:
print(f"B2_Mm2 with CTCF:    {len(sineB2_with_ctcf):,}")
print(f"B2_Mm2 without CTCF: {len(sineB2_no_ctcf):,}")

B2_Mm2 with CTCF:    51,779
B2_Mm2 without CTCF: 47,165


In [15]:
standard_chroms = [f"chr{i}" for i in range(1, 20)] + ["chrX", "chrY"]

In [16]:
sineB2_with_ctcf = sineB2_with_ctcf[sineB2_with_ctcf["genoName"].isin(standard_chroms)]
sineB2_no_ctcf = sineB2_no_ctcf[sineB2_no_ctcf["genoName"].isin(standard_chroms)]

In [17]:
sineB2_with_ctcf_300 = sineB2_with_ctcf.sample(300, random_state=42)
sineB2_no_ctcf_300 = sineB2_no_ctcf.sample(300, random_state=42)

In [19]:
for name, df in [("with CTCF", sineB2_with_ctcf_300), ("without CTCF", sineB2_no_ctcf_300)]:
    lengths = df["genoEnd"] - df["genoStart"]
    print(f"B2_Mm2 {name}: max={lengths.max()}, min={lengths.min()}, median={lengths.median():.0f}")

B2_Mm2 with CTCF: max=219, min=51, median=181
B2_Mm2 without CTCF: max=211, min=11, median=145


In [20]:
sineB2_with_ctcf_300.to_csv("sineB2_with_ctcf_300.tsv", sep="\t", index=False)
sineB2_no_ctcf_300.to_csv("sineB2_no_ctcf_300.tsv", sep="\t", index=False)

print(f"Exported: {len(sineB2_with_ctcf_300)} with CTCF, {len(sineB2_no_ctcf_300)} without CTCF")

Exported: 300 with CTCF, 300 without CTCF
